In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [ ]:
ground_truth[10]

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [ ]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [ ]:
q = ground_truth[10]
q

In [ ]:
doc_idx[q['document']]

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
)

In [ ]:
q['question']

In [ ]:
answer = assistant.rag(q['question'])

In [ ]:
assistant.total_cost()

In [ ]:
print(answer)

In [ ]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

In [ ]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

In [ ]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [ ]:
record = generate_rag_answer(q)
record

In [ ]:
assistant.total_cost()

In [ ]:
assistant.reset_usage()

In [ ]:
assistant.total_cost()

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

In [ ]:
results[:10]

In [ ]:
df_results = pd.DataFrame(results)

In [ ]:
df_results.head()

In [ ]:
assistant.total_cost()


In [ ]:
df_results.to_csv("data/rag-answers-new.csv", index=False)